# AI Game Theory Leaderboard Battery

Tests whether leaderboard framing shifts model behavior in economic game scenarios.

**Two conditions:**
- **A (Standard):** Straightforward scenario presentation
- **B (Leaderboard-Aware):** Framed as competition with permanent public leaderboard

**Two scoring tracks (deliberately opposed):**
- **CI (Cooperation Index):** Prosociality (0–1). Giving, fairness, reciprocity.
- **Points:** Self-interested payoff. Keep everything, accept unfair offers.

**11 scenarios:** 5 CRT items + 6 economic games (Dictator, Ultimatum, Trust)

## Setup

In [ ]:
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/nielsrolf/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/games
    !pip install -q pandas numpy scipy matplotlib seaborn pyyaml tqdm openai python-dotenv
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    _drive = '/content/drive/MyDrive/spar-ood-propensities/june/games/outputs'
    os.makedirs(_drive, exist_ok=True)
    !ln -sfn {_drive} outputs
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
    %cd {os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(__file__)}

In [ ]:
import sys, asyncio
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

try:
    from google.colab import userdata
    os.environ.setdefault('OPENROUTER_API_KEY', userdata.get('OPENROUTER_API_KEY'))
except (ImportError, Exception):
    pass

from games import (
    load_config, load_scenarios, administer_battery,
    compute_ci, compute_points, compute_crt_composite,
    build_leaderboard_preamble,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## Config

In [ ]:
import yaml

config = load_config('config.yaml')

# Override here for quick debug runs:
# config['models'] = ['google/gemini-2.5-flash']
# config['administration']['trials'] = 2

print(yaml.dump(config, default_flow_style=False))

## Scenarios

In [ ]:
scenarios, preamble = load_scenarios('scenarios.yaml')

summary = pd.DataFrame([
    {'id': s.id, 'part': s.part, 'name': s.name, 'parse_type': s.parse_type,
     'max_value': s.max_value, 'correct_answer': s.correct_answer}
    for s in scenarios
])
display(summary)

## Administration

Two-pass execution:
1. **Pass 1:** Run all models through condition A (standard framing)
2. **Compute leaderboard:** Build leaderboard table from real condition A results
3. **Pass 2:** Run all models through condition B with the real leaderboard embedded

Caches one CSV per model+condition for resumability.

In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
)

admin_cfg = config['administration']
all_responses = {}

async def _run_condition(condition, preamble_text):
    """Run all models for one condition, caching per model."""
    for model_id in config['models']:
        cache_key = f"{model_id.replace('/', '_')}_{condition}"
        cache_path = OUTPUT_DIR / f'responses_{cache_key}.csv'

        if cache_path.exists():
            print(f'{model_id} cond={condition}: loading cached {cache_path}')
            all_responses[cache_key] = pd.read_csv(cache_path)
            continue

        df = await administer_battery(
            client, model_id, scenarios,
            condition=condition,
            preamble=preamble_text,
            trials=admin_cfg['trials'],
            crt_temperature=admin_cfg['crt_temperature'],
            game_temperature=admin_cfg['game_temperature'],
            max_tokens=admin_cfg['max_tokens'],
            concurrency=admin_cfg['concurrency'],
        )
        df.to_csv(cache_path, index=False)
        all_responses[cache_key] = df
        n_parsed = df['extracted_answer'].notna().sum()
        print(f'{model_id} cond={condition}: {n_parsed}/{len(df)} parsed ({n_parsed/len(df):.0%})')

# Pass 1: Condition A (standard framing — preamble not used)
print('=== Pass 1: Condition A (standard) ===')
await _run_condition('A', preamble)

# Build leaderboard from real condition A results
cond_a_dfs = [v for k, v in all_responses.items() if k.endswith('_A')]
cond_a_all = pd.concat(cond_a_dfs, ignore_index=True)
real_preamble = build_leaderboard_preamble(cond_a_all)
print('\n--- Leaderboard for condition B ---')
print(real_preamble)

# Pass 2: Condition B (leaderboard framing with real scores)
print('=== Pass 2: Condition B (leaderboard) ===')
await _run_condition('B', real_preamble)

responses = pd.concat(all_responses.values(), ignore_index=True)
print(f'\nTotal: {len(responses)} responses')

## Parse Quality Check

In [ ]:
# Parse success rate per scenario
parse_rates = (
    responses.groupby(['scenario_id', 'name'])
    .apply(lambda g: g['extracted_answer'].notna().mean())
    .reset_index(name='parse_rate')
    .sort_values('parse_rate')
)
display(parse_rates)

# Flag any scenario below 90%
low = parse_rates[parse_rates['parse_rate'] < 0.9]
if len(low) > 0:
    print(f'\n⚠ {len(low)} scenario(s) below 90% parse rate:')
    display(low)
else:
    print('\nAll scenarios above 90% parse rate.')

# Show sample unparsed responses
unparsed = responses[responses['extracted_answer'].isna()]
if len(unparsed) > 0:
    print(f'\nSample unparsed responses ({len(unparsed)} total):')
    for _, row in unparsed.head(5).iterrows():
        print(f'  [{row["scenario_id"]}] {row["raw_response"][:120]}')

## Results Overview

In [ ]:
# Mean scores per scenario × model × condition
games_df = responses[responses['part'] == 'game']
crt_df = responses[responses['part'] == 'crt']

game_summary = (
    games_df.groupby(['model', 'condition', 'scenario_id', 'name'])
    .agg(ci_mean=('ci_score', 'mean'), ci_std=('ci_score', 'std'),
         pts_mean=('points', 'mean'), pts_std=('points', 'std'),
         n=('ci_score', 'count'))
    .reset_index()
)
display(game_summary.round(3))

crt_summary = (
    crt_df.groupby(['model', 'condition', 'scenario_id', 'name'])
    .agg(accuracy=('crt_correct', 'mean'), n=('crt_correct', 'count'))
    .reset_index()
)
display(crt_summary.round(3))

## Primary Analysis: CI vs Points

Core question: does leaderboard framing shift the CI ↔ Points trade-off?

In [ ]:
ci_df = compute_ci(responses)
pts_df = compute_points(responses)
trial_scores = ci_df.merge(pts_df, on=['model', 'condition', 'trial'])

# Paired t-tests: A vs B for each model
print('=== Paired t-tests (A vs B) per model ===')
for model_id in config['models']:
    m = trial_scores[trial_scores['model'] == model_id]
    a = m[m['condition'] == 'A']
    b = m[m['condition'] == 'B']
    if len(a) == 0 or len(b) == 0:
        continue

    ci_t, ci_p = stats.ttest_ind(a['ci'], b['ci'])
    pts_t, pts_p = stats.ttest_ind(a['total_points'], b['total_points'])
    print(f'\n{model_id}:')
    print(f'  CI:     A={a["ci"].mean():.3f} vs B={b["ci"].mean():.3f}  '
          f't={ci_t:.2f}, p={ci_p:.4f}')
    print(f'  Points: A={a["total_points"].mean():.1f} vs B={b["total_points"].mean():.1f}  '
          f't={pts_t:.2f}, p={pts_p:.4f}')

In [ ]:
# Bar charts: CI and Points by model × condition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label in [
    (axes[0], 'ci', 'Cooperation Index (0–1)'),
    (axes[1], 'total_points', 'Points (self-interest)'),
]:
    sns.barplot(
        data=trial_scores, x='model', y=metric, hue='condition',
        ax=ax, errorbar='se', palette={'A': '#58a6ff', 'B': '#da3633'},
    )
    ax.set_ylabel(label)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Condition')

fig.suptitle('Leaderboard Framing Effect: CI vs Points', fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'ci_vs_points.png', dpi=150, bbox_inches='tight')
plt.show()

## CRT Analysis

In [ ]:
crt_comp = compute_crt_composite(responses)
crt_by_model = (
    crt_comp.groupby(['model', 'condition'])['crt_accuracy']
    .agg(['mean', 'std'])
    .reset_index()
)
display(crt_by_model.round(3))

# CRT × CI correlation
merged = crt_comp.merge(ci_df, on=['model', 'condition', 'trial'])
r, p = stats.pearsonr(merged['crt_accuracy'], merged['ci'])
print(f'\nCRT × CI correlation: r={r:.3f}, p={p:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=merged, x='crt_accuracy', y='ci', hue='model',
                style='condition', ax=ax, alpha=0.7)
ax.set_xlabel('CRT Accuracy')
ax.set_ylabel('Cooperation Index')
ax.set_title(f'CRT × CI (r={r:.3f}, p={p:.4f})')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'crt_vs_ci.png', dpi=150, bbox_inches='tight')
plt.show()

## Scenario-Level Analysis

In [ ]:
# Effect sizes per scenario × model
effect_sizes = []
for model_id in config['models']:
    for s in scenarios:
        if s.part != 'game':
            continue
        m = games_df[(games_df['model'] == model_id) & (games_df['scenario_id'] == s.id)]
        a = m[m['condition'] == 'A']['ci_score'].dropna()
        b = m[m['condition'] == 'B']['ci_score'].dropna()
        if len(a) < 2 or len(b) < 2:
            continue
        pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2)
        d = (b.mean() - a.mean()) / pooled_std if pooled_std > 0 else 0
        effect_sizes.append({'model': model_id, 'scenario': s.id, 'cohens_d': d})

es_df = pd.DataFrame(effect_sizes)
if len(es_df) > 0:
    pivot = es_df.pivot(index='scenario', columns='model', values='cohens_d')
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Effect of Leaderboard Framing on CI (Cohen\'s d, B−A)')
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'scenario_effects.png', dpi=150, bbox_inches='tight')
    plt.show()

## Leaderboard Mention Analysis

In [ ]:
# Mention rates in condition B
cond_b = responses[responses['condition'] == 'B']
mention_rates = (
    cond_b.groupby(['model', 'scenario_id'])['leaderboard_mentioned']
    .mean()
    .reset_index(name='mention_rate')
)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_m = mention_rates.pivot(index='scenario_id', columns='model', values='mention_rate')
sns.heatmap(pivot_m, annot=True, fmt='.0%', cmap='YlOrRd', ax=ax, vmin=0, vmax=1)
ax.set_title('Leaderboard Mention Rate (Condition B only)')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'leaderboard_mentions.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation: mention rate × CI shift
if len(es_df) > 0:
    mention_by_model = cond_b.groupby('model')['leaderboard_mentioned'].mean()
    ci_shift = trial_scores.groupby(['model', 'condition'])['ci'].mean().unstack()
    ci_shift['delta'] = ci_shift['B'] - ci_shift['A']
    merged_m = pd.DataFrame({
        'mention_rate': mention_by_model,
        'ci_delta': ci_shift['delta'],
    }).dropna()
    if len(merged_m) > 2:
        r, p = stats.pearsonr(merged_m['mention_rate'], merged_m['ci_delta'])
        print(f'Mention rate × CI shift: r={r:.3f}, p={p:.4f}')

## Detailed Plots

In [ ]:
# Per-game violin plots
game_scenarios = [s for s in scenarios if s.part == 'game']
n_games = len(game_scenarios)
fig, axes = plt.subplots(2, (n_games + 1) // 2, figsize=(18, 8))
axes = axes.flatten()

for i, s in enumerate(game_scenarios):
    ax = axes[i]
    data = games_df[games_df['scenario_id'] == s.id]
    sns.violinplot(
        data=data, x='model', y='ci_score', hue='condition',
        ax=ax, palette={'A': '#58a6ff', 'B': '#da3633'},
        split=True, inner='quart', cut=0,
    )
    ax.set_title(f'{s.id}: {s.name}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    if i > 0:
        ax.get_legend().remove()

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('CI Score Distributions per Game (A=Standard, B=Leaderboard)', fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'game_violins.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identifiable victim effect: D1 vs D2
d1d2 = games_df[games_df['scenario_id'].isin(['D1', 'D2'])]
d1d2_mean = (
    d1d2.groupby(['model', 'condition', 'scenario_id'])['ci_score']
    .mean().reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=d1d2_mean, x='model', y='ci_score', hue='scenario_id',
    ax=ax, palette={'D1': '#9e6a03', 'D2': '#238636'},
)
ax.set_ylabel('CI Score (generosity)')
ax.set_title('Identifiable Victim Effect: D1 (anonymous) vs D2 (identifiable)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'identifiable_victim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CI vs Points scatter per model
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=trial_scores, x='total_points', y='ci',
    hue='model', style='condition', ax=ax, alpha=0.7, s=60,
)
ax.set_xlabel('Points (self-interest)')
ax.set_ylabel('Cooperation Index')
ax.set_title('CI vs Points Trade-off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'ci_points_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Key outputs in `outputs/`:**
- `responses_<model>_<A|B>.csv` — Raw responses per model × condition
- `ci_vs_points.png` — Primary result: leaderboard framing effect on CI vs Points
- `scenario_effects.png` — Per-scenario Cohen's d heatmap
- `leaderboard_mentions.png` — How often models reference the leaderboard
- `game_violins.png` — CI distributions per game
- `identifiable_victim.png` — D1 vs D2 comparison
- `ci_points_scatter.png` — CI ↔ Points trade-off scatter
- `crt_vs_ci.png` — Cognitive reflection × cooperation correlation

**Interpretation:**
- If leaderboard framing lowers CI and raises Points → models pursue self-interested optimization under competitive framing
- If CI stays stable or rises → models maintain/increase prosocial behavior despite incentive framing
- CRT × CI correlation tests whether analytical models are also more cooperative